In [ ]:
import pandas as pd
import torch
import torchvision
import torchvision.transforms.v2 as transforms_v2
import pydicom
import numpy as np
import os
from pathlib import Path
import optuna
import metric
from collections import defaultdict
import matplotlib.pyplot as plt
from torch.utils.tensorboard import SummaryWriter
import time
import glob
import pickle

# import torch.multiprocessing as mp
# mp.set_start_method('spawn', force=True)

# Set random seeds for reproducibility
np.random.seed(42) # TODO update (current use is legacy and not effective)
torch.manual_seed(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(DEVICE, torch.cuda.is_available())

In [ ]:
def extract_slice_positions(series_dir, multi_block):
    """
    Parse all DICOMs in a series, return ordered slice list by position along normal axis.
    Returns:
        ordered_ids: [dcm_id_1, dcm_id_2, ...] sorted along slice axis
    """
    slice_infos = []
    
    for fname in os.listdir(series_dir):
        full = os.path.join(series_dir, fname)
        if full in multi_block:
            continue

        dcm_id = Path(fname).stem
        try:
            ds = pydicom.dcmread(
                full, stop_before_pixels=True, force=True,
                specific_tags=["ImagePositionPatient", "ImageOrientationPatient"]
            )
            ipp = np.array(ds.ImagePositionPatient, dtype=float)  # (x,y,z)
            iop = np.array(ds.ImageOrientationPatient, dtype=float)  # 6 values

            # Normal vector = cross product of row & column direction cosines
            row_cos = iop[:3]
            col_cos = iop[3:]
            normal = np.cross(row_cos, col_cos)

            # Project ipp onto normal
            pos = float(np.dot(ipp, normal))

            slice_infos.append((dcm_id, pos))
        except Exception as e:
            # Skip problematic files
            continue

    # Sort by position
    slice_infos.sort(key=lambda x: x[1])
    ordered_ids = [d[0] for d in slice_infos]
    return ordered_ids


def mark_positive_with_neighbors(series_id, loc_ids, classification_df, aneurysm, ordered_ids):
    """
    Mark positive frames and their Â±1 neighbors based on sorted slice order.
    """
    id_to_idx = {dcm_id: idx for idx, dcm_id in enumerate(ordered_ids)}
    
    for loc_id in loc_ids:
        if (series_id, loc_id) not in classification_df:
            continue
        # mark central
        classification_df[(series_id, loc_id)] = aneurysm

        if loc_id not in id_to_idx:
            print(f"Warning: {loc_id} not found in ordered_ids for series {series_id}")
            continue

        idx = id_to_idx[loc_id]
        for nb_idx in [idx - 1, idx + 1]:
            if 0 <= nb_idx < len(ordered_ids):
                nb_id = ordered_ids[nb_idx]
                key = (series_id, nb_id)
                if key in classification_df:
                    classification_df[key] = aneurysm
    
                    
def show_series_slices(series_id, dcm_ids, series_path, ncols=3):
    """Plot given slices from a series to visually check adjacency."""
    plt.figure(figsize=(4*ncols, 4))
    
    for i, dcm_id in enumerate(dcm_ids):
        dcm_path = os.path.join(series_path, series_id, f"{dcm_id}.dcm")
        ds = pydicom.dcmread(dcm_path, stop_before_pixels=False)
        img = ds.pixel_array
        
        plt.subplot(1, len(dcm_ids), i+1)
        plt.imshow(img, cmap="gray")
        plt.title(f"Slice {dcm_id}")
        plt.axis("off")
    
    plt.tight_layout()
    plt.show()

In [ ]:
root_path = '/kaggle/input/rsna-intracranial-aneurysm-detection'
series_path = os.path.join(root_path, 'series')

train_df = pd.read_csv(os.path.join(root_path,'train.csv'))
train_loc_df = pd.read_csv(os.path.join(root_path,'train_localizers.csv'))

multi_dicom_df = pd.read_csv('/kaggle/input/mlt-frame-dcm/mlt_frame_dcm_files_2.csv')
multi_block = set(multi_dicom_df['dicom_path'].astype(str))

classification_df = {}
index_map = []  # (series_id, dcm_id) -> index in classification_df
series_pos_maps = {}  # series_id -> ordered_ids
# build dict: series_id -> list of loc_ids
loc_map = train_loc_df.astype(str).groupby(train_loc_df.columns[0])[train_loc_df.columns[1]].apply(list).to_dict()

if os.path.exists("/kaggle/input/series-files/series_files.pkl"):
    with open("/kaggle/input/series-files/series_files.pkl", "rb") as f:
        series_files = pickle.load(f)
else:
    all_files = glob.glob(os.path.join(series_path, "*", "*"), recursive=True)
    series_files = defaultdict(list)
    for f in all_files:
        series_id = Path(f).parent.name
        series_files[series_id].append(f)
    with open("series_files.pkl", "wb") as f:
        pickle.dump(series_files, f)

i = 0
breakpoint = 10
count = 0
for row in train_df.itertuples(index=False):
    if i%100 == 0: print(i)
    i += 1
    series_id = str(row[0])
    aneurysm = np.array(row[-14:], dtype=int)

    series_dir = os.path.join(series_path, series_id)
    if not os.path.isdir(series_dir):
        continue
    ordered_ids = extract_slice_positions(series_dir, multi_block)

    series_pos_maps[series_id] = ordered_ids

    entries = {}
    for full_path in series_files[series_id]:
        if full_path in multi_block:
            continue
        dcm_id = Path(full_path).stem
        entries[(series_id, dcm_id)] = np.zeros(14, dtype=int)
        index_map.append((series_id, dcm_id))

    classification_df.update(entries)

    if aneurysm[-1] == 1:
        count += 1
        loc_ids = loc_map.get(series_id, [])
        mark_positive_with_neighbors(series_id, loc_ids, classification_df, aneurysm, ordered_ids)
    if count >= breakpoint:
        break
# Sanity checks
assert len(classification_df) == len(index_map)
print("Exemple dtype:", type(list(classification_df.values())[1][0]))

positive_series = list(set(sid for (sid, did), y in classification_df.items() if y[-1] == 1))
print("Positive series:", len(positive_series), "/", len(series_pos_maps))
negative_series = list(set(series_pos_maps.keys()) - set(positive_series))
print("Negative series:", len(negative_series), "/", len(series_pos_maps))

# split train/val using balanced sampling series wise with a 20% positive in val (val has 50% pos/neg)
val_indices = []
train_indices = []
# shuffle series
np.random.shuffle(positive_series)
np.random.shuffle(negative_series)
# take first 20% of positive series for val
n_val_pos = max(1, int(0.2 * len(positive_series)))
val_pos_series = positive_series[:n_val_pos]
train_pos_series = positive_series[n_val_pos:]
# take same number of negative series for val
n_val_neg = n_val_pos
val_neg_series = negative_series[:n_val_neg]
train_neg_series = negative_series[n_val_neg:]
print(f"Val: {len(val_pos_series)} positive series, {len(val_neg_series)} negative series")
# assign indices    
for i, (sid, did) in enumerate(index_map):
    if sid in val_pos_series or sid in val_neg_series:
        val_indices.append(i)
    else:
        train_indices.append(i)
        
# get the total nb of pos and neg respectively in train and val
n_train_pos = sum(1 for (sid, did), y in classification_df.items() 
                  if y[-1] == 1 and (sid in train_pos_series or sid in train_neg_series))
n_train_neg = sum(1 for (sid, did), y in classification_df.items() 
                  if y[-1] == 0 and (sid in train_pos_series or sid in train_neg_series))
n_val_pos = sum(1 for (sid, did), y in classification_df.items()
                if y[-1] == 1 and (sid in val_pos_series or sid in val_neg_series))
n_val_neg = sum(1 for (sid, did), y in classification_df.items()
                if y[-1] == 0 and (sid in val_pos_series or sid in val_neg_series))
print(f"Train: {n_train_pos} positive, {n_train_neg} negative")
print(f"Val: {n_val_pos} positive, {n_val_neg} negative") 

In [ ]:
# TODO: Transforms 
train_transforms_1 = transforms_v2.Compose([
    transforms_v2.Resize((512, 512)),
    transforms_v2.RandomRotation(degrees=15),
    transforms_v2.RandomHorizontalFlip(),
    transforms_v2.ColorJitter(brightness=0.2, contrast=0.2),
])
val_transforms = transforms_v2.Compose([
    transforms_v2.Resize((512, 512)),
])

class AneurysmDataset(torch.utils.data.Dataset):
    """Dataset for RSNA Intracranial Aneurysm Detection.
    Args:
        root_path (str): Path to the dataset root directory.
        train (bool): If True, use the training set; if False, use the validation set.
    """
    def __init__(self, root_path, train=True, transforms=None):
        super().__init__()
        self.root_path = root_path
        self.classification_df = classification_df
        self.train = train
        self.transforms = transforms
        self.pixel_cache = {}
        # create index map based on train or validation indices
        if train:
            self.index_map = {new_i: index_map[idx] for new_i, idx in enumerate(train_indices)}
        else:
            # create index map for validation set with all positive validation samples and equal number of negative samples
            self.index_map = {}
            for sid in val_pos_series:
                for (s, d), y in classification_df.items():
                    if s == sid and y[-1] == 1 :
                        self.index_map[len(self.index_map)] = (s, d)
            count_neg = 0
            for sid in val_neg_series:
                for (s, d), y in classification_df.items():
                    if s == sid and y[-1] == 0 and count_neg < n_val_pos:
                        self.index_map[len(self.index_map)] = (s, d)
                        count_neg += 1
                    if count_neg >= n_val_pos:
                        break
                if count_neg >= n_val_pos:
                    break

    def __getitem__(self, index):
        img_id = self.index_map[index] 
        img_path = os.path.join(self.root_path, 'series', img_id[0], img_id[1]+'.dcm')
        if img_path in self.pixel_cache:
            img_array = self.pixel_cache[img_path]
        else:
            # start_read = time.time()
            img_array = pydicom.dcmread(img_path).pixel_array
            # print(f"Read DICOM time: {time.time() - start_read:.4f} seconds")
            self.pixel_cache[img_path] = img_array
        # start_tensor_trans = time.time()
        img = torch.tensor(img_array, dtype=torch.float).unsqueeze(0)
        if self.transforms is not None:
            img = self.transforms(img)
        # print(f"Tensor and transforms time: {time.time() - start_tensor_trans:.4f} seconds")
        # start_device = time.time()
        img = img.to(DEVICE)
        # print(f"To device time: {time.time() - start_device:.4f} seconds")
        target = torch.tensor(self.classification_df[img_id].astype(int), dtype=torch.float, device=DEVICE)
        return img, target

    def __len__(self):
        return len(self.index_map.keys())
    
class SeriesAneurysmDataset(torch.utils.data.Dataset):
    """
    Dataset for all slices in a single series for validation/testing.
    Loads all DICOM slices for a given series_id, applies transforms, and provides (img, target) pairs.
    """
    def __init__(self, series_id, series_path, ordered_ids, classification_df, transforms=None, device='cpu'):
        self.series_id = series_id
        self.series_path = series_path
        self.ordered_ids = ordered_ids
        self.classification_df = classification_df
        self.transforms = transforms
        self.device = device

        # Filter only slices present in classification_df
        self.slice_ids = [dcm_id for dcm_id in ordered_ids if (series_id, dcm_id) in classification_df]

    def __len__(self):
        return len(self.slice_ids)

    def __getitem__(self, idx):
        dcm_id = self.slice_ids[idx]
        img_path = os.path.join(self.series_path, self.series_id, f"{dcm_id}.dcm")
        img_array = pydicom.dcmread(img_path).pixel_array
        img = torch.tensor(img_array, dtype=torch.float).unsqueeze(0)  # (1, H, W)
        if self.transforms is not None:
            img = self.transforms(img)
        img = img.to(self.device)
        target = torch.tensor(self.classification_df[(self.series_id, dcm_id)].astype(int), dtype=torch.float, device=self.device)
        return img, target

def make_train_loader(root_path, alpha=2, batch_size=32):
    """ Create a DataLoader for the training set with balanced classes.
    Args:
        root_path (str): Path to the dataset root directory.
        alpha (float): Weighting factor for the negative class sampling.
        The higher the value, the more negative samples are included in each epoch.
        batch_size (int): Size of the batches to be returned by the DataLoader.
    Returns:
        torch.utils.data.DataLoader: DataLoader for the training set with balanced classes.
    """
    dataset = AneurysmDataset(root_path, train=True, transforms=train_transforms_1)
    # sampler to ensure balanced classes in each batch
    sampler = torch.utils.data.WeightedRandomSampler(
        weights=[1.0 / n_train_pos if dataset.classification_df[dataset.index_map[i]][-1] == 1 else alpha / n_train_neg for i in range(len(dataset))],
        num_samples=n_train_pos*3, # reduce dataset size to avoid too much resampling
        replacement=True
    )
    return torch.utils.data.DataLoader(
        dataset,
        batch_size=batch_size,
        sampler=sampler,
    )

In [ ]:
#Train loop

def train_one_epoch(model, dataloader, optimizer, loss_fn):
    model.train()
    running_loss = 0.0
    total_infer_time = 0e-6
    total_loss_time = 0e-6
    total_dataload_time = 0e-6
    total_samples = 0
    start_dataload = time.time()
    for images, labels in dataloader:
        total_dataload_time += time.time() - start_dataload
        optimizer.zero_grad()
        start_infer = time.time()
        preds = model(images)
        total_infer_time += time.time() - start_infer
        start_loss = time.time()
        loss = loss_fn(preds,labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        total_loss_time += time.time() - start_loss
        total_samples += images.size(0)
        start_dataload = time.time()
    avg_loss = running_loss / len(dataloader)
    print(f"Training loss: {avg_loss:.4f} over {total_samples} samples", len(dataloader.sampler))
    print(f"  - Data loading time: {total_dataload_time:.2f} seconds")
    print(f"  - Model forward time: {total_infer_time:.2f} seconds")
    print(f"  - Loss computation time: {total_loss_time:.2f} seconds")
    return avg_loss
    
    
def test_model(model, dataloader, loss_fn):
    model.eval()
    preds_list = []
    labels_list = []
    total_batch_time = 0e-6
    total_append_time = 0e-6
    total_dataload_time = 0e-6
    start_infer = time.time()
    with torch.no_grad():
        start_dataload = time.time()
        for images, labels in dataloader:
            total_dataload_time += time.time() - start_dataload
            start_batch = time.time()
            batch_preds = model(images)
            total_batch_time += time.time() - start_batch
            start_append = time.time()
            preds_list.append(batch_preds)
            labels_list.append(labels)
            total_append_time += time.time() - start_append
            start_dataload = time.time()
    print(f"Inference time: {time.time() - start_infer:.2f} seconds")
    print(f"  - Model forward time: {total_batch_time:.2f} seconds")
    print(f"  - Append time: {total_append_time:.2f} seconds")
    print(f"  - Data loading time: {total_dataload_time:.2f} seconds")
    # Concatenate once
    all_preds = torch.cat(preds_list, dim=0)
    all_labels = torch.cat(labels_list, dim=0)
    print("Total samples:", all_labels.shape[0])
    # Compute loss
    loss = loss_fn(all_preds, all_labels)

    # Apply sigmoid to predictions
    all_preds = torch.sigmoid(all_preds)

    y_true = all_labels.cpu().numpy()
    y_score = all_preds.cpu().numpy()

    # Original weights
    class_weights = np.array([1,1,1,1,1,1,1,1,1,1,1,1,1,13], dtype=float)

    # Detect valid classes
    valid_mask = np.array([len(np.unique(y_true[:, j])) == 2 for j in range(y_true.shape[1])])
    valid_classes = np.where(valid_mask)[0].tolist()
    ignored_classes = np.where(~valid_mask)[0].tolist()

    print(f"Ignored classes (only 0s or only 1s): {ignored_classes}")
    print(f"Used classes: {valid_classes}")

    if len(valid_classes) == 0:
        print("No valid classes for ROC AUC.")
        comp_metric = np.nan
    else:
        # Filter inputs and weights
        y_true_valid = y_true[:, valid_classes]
        y_score_valid = y_score[:, valid_classes]
        class_weights_valid = class_weights[valid_classes]

        # Call your existing metric function
        comp_metric = metric.weighted_multilabel_auc(
            y_true_valid, 
            y_score_valid, 
            class_weights_valid
        )

    return loss, comp_metric


def test_model_per_series(model, series_path, series_pos_maps, classification_df, batch_size=32):
    """
    Test the model on each series and compute average loss and global metric (AUC ROC).
    For ROC AUC, use the max prediction for each class in a series.
    Args:
        model: Trained model to evaluate.
        series_path: Path to the series directory.
        series_pos_maps: Dictionary mapping series_id to ordered slice IDs.
        classification_df: Dictionary mapping (series_id, dcm_id) to target vectors.
        batch_size: Number of slices to process at once.
    Returns:
        global_metric: Global metric (AUC ROC) across all series.
    """
    model.eval()
    all_series_preds = []
    all_series_labels = []
    total_slices = 0

    # Only evaluate on validation series
    val_series_set = set(val_pos_series) | set(val_neg_series)

    with torch.no_grad():
        for series_id in val_series_set:
            ordered_ids = series_pos_maps.get(series_id, [])
            if not ordered_ids:
                continue
            dataset = SeriesAneurysmDataset(series_id, series_path, ordered_ids, classification_df, transforms=val_transforms, device=DEVICE)
            loader = torch.utils.data.DataLoader(dataset, batch_size=batch_size)
            preds_list = []
            labels_list = []
            total_series_slices = 0

            for batch_imgs, batch_labels in loader:
                batch_imgs = batch_imgs.to(DEVICE)
                batch_labels = batch_labels.to(DEVICE)
                batch_preds = model(batch_imgs)
                preds_list.append(batch_preds.cpu())
                labels_list.append(batch_labels.cpu())
                total_series_slices += batch_imgs.size(0)

            if not preds_list:
                continue

            preds = torch.cat(preds_list, dim=0)  # (num_slices, num_classes)
            series_labels = torch.cat(labels_list, dim=0)  # (num_slices, num_classes)

            # For ROC AUC: take max prediction per class for the series
            max_preds = torch.max(torch.sigmoid(preds), dim=0)[0]  # (num_classes,)
            max_labels = torch.max(series_labels, dim=0)[0]         # (num_classes,)

            all_series_preds.append(max_preds.numpy())
            all_series_labels.append(max_labels.numpy())
            total_slices += total_series_slices

    # Stack all series-level predictions and labels
    y_true = np.stack(all_series_labels, axis=0)
    y_score = np.stack(all_series_preds, axis=0)

    # Original weights
    class_weights = np.array([1,1,1,1,1,1,1,1,1,1,1,1,1,13], dtype=float)

    # Detect valid classes
    valid_mask = np.array([len(np.unique(y_true[:, j])) == 2 for j in range(y_true.shape[1])])
    valid_classes = np.where(valid_mask)[0].tolist()
    ignored_classes = np.where(~valid_mask)[0].tolist()
    print(f"Ignored classes (only 0s or only 1s): {ignored_classes}")
    print(f"Used classes: {valid_classes}")
    if len(valid_classes) == 0:
        print("No valid classes for ROC AUC.")
        global_metric = np.nan
    else:
        # Filter inputs and weights
        y_true_valid = y_true[:, valid_classes]
        y_score_valid = y_score[:, valid_classes]
        class_weights_valid = class_weights[valid_classes]

        global_metric = metric.weighted_multilabel_auc(
            y_true_valid, 
            y_score_valid, 
            class_weights_valid
        )
    return global_metric

def set_finetune_params(model, n_top_layers=2):
    # Freeze all backbone layers
    for param in model.backbone.parameters():
        param.requires_grad = False
    # Unfreeze classifier
    for param in model.classifier.parameters():
        param.requires_grad = True
    # Unfreeze last n_top_layers of backbone
    for layer in list(model.backbone.children())[-n_top_layers:]:
        for param in layer.parameters():
            param.requires_grad = True

In [ ]:
# Model

class AneurysmClassifier(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = torchvision.models.efficientnet_b3(weights="DEFAULT").features
        self.avgpool = torch.nn.AdaptiveAvgPool2d(1)
        self.classifier = torch.nn.Sequential(
            torch.nn.Dropout(0.3),
            torch.nn.Linear(1536, 256), # 1536 is the number of features for efficientnet_b3
            torch.nn.ReLU(),
            torch.nn.Linear(256, 1)
        )
        self.loc_classifier = torch.nn.Sequential(
            torch.nn.Dropout(0.3),
            torch.nn.Linear(1536, 256), # 1536 is the number of features for efficientnet_b3
            torch.nn.ReLU(),
            torch.nn.Linear(256, 13)
        )
    def forward(self, x):
        # make sure input has 3 channels
        if x.size(1) == 1:
            x = x.repeat(1, 3, 1, 1)
        x = self.backbone(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        loc = self.loc_classifier(x)
        present = self.classifier(x)
        return torch.cat((loc,present), dim=-1)        

In [ ]:
# Loss function
class CustomLoss(torch.nn.Module):
    """
    Compute BCE loss for presence and CE loss for location.
    Args:
        pos_weight (float or tensor): Weight for positive class in BCE loss.
    """
    def __init__(self, pos_weight=None):
        super().__init__()
        self.bce_loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        self.bce_loss_fn_loc = torch.nn.BCEWithLogitsLoss()

    def forward(self, preds, labels):
        presence_preds, loc_preds = preds[:, -1].unsqueeze(1), preds[:, :-1]
        presence_labels, loc_labels = labels[:, -1].unsqueeze(1), labels[:, :-1]

        presence_loss = self.bce_loss_fn(presence_preds, presence_labels)

        loc_loss = self.bce_loss_fn_loc(loc_preds, loc_labels)
        
        total_loss = presence_loss + loc_loss
        return total_loss


In [ ]:

# Define hyperparameters to tune
learning_rate = 5e-5
loss_pos_weight_alpha = 1.
sampling_weight_alpha = 2.
exp_scheduler = 0.98

# Initialize model, optimizer, scheduler, and loss function
model = AneurysmClassifier()
model.to(DEVICE)
pos_weights = torch.tensor((n_train_neg / n_train_pos) * loss_pos_weight_alpha, dtype=torch.float)
set_finetune_params(model, n_top_layers=2)
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=learning_rate)
scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=exp_scheduler)
loss_fn = CustomLoss(pos_weight=pos_weights)
val_loss_fn = CustomLoss()
train_loader = make_train_loader('/kaggle/input/rsna-intracranial-aneurysm-detection', alpha=sampling_weight_alpha, batch_size=32)
val_loader = torch.utils.data.DataLoader(
    AneurysmDataset('/kaggle/input/rsna-intracranial-aneurysm-detection', train=False, transforms=val_transforms),
    batch_size=32,
)

epochs = 2
# Train the model
writer = SummaryWriter(log_dir='runs/aneurysm_experiment_1')
for epoch in range(epochs):
    print(f"Epoch {epoch+1}/{epochs}")
    start_train = time.time()
    avg_loss = train_one_epoch(model, train_loader, optimizer, loss_fn)
    scheduler.step()
    print(f"Training time for epoch {epoch+1}: {time.time() - start_train:.2f} seconds")
    print(f"Completed epoch {epoch+1}")
    # Test the model on validation set
    test_time = time.time()
    loss, comp_metric = test_model(model, val_loader, val_loss_fn)
    print(loss, comp_metric)
    print(f"Testing time for epoch {epoch+1}: {time.time() - test_time:.2f} seconds")
    test_time = time.time()
    series_comp_metric = test_model_per_series(model, series_path, series_pos_maps, classification_df, batch_size=32)
    print(series_comp_metric)
    print(f"Series testing time for epoch {epoch+1}: {time.time() - test_time:.2f} seconds")
    
    # TensorBoard logging
    writer.add_scalar('Train/Loss', avg_loss, epoch)
    writer.add_scalar('Loss/val_batch', loss, epoch)
    writer.add_scalar('Metric/val_batch', comp_metric, epoch)
    writer.add_scalar('Metric/val_series', series_comp_metric, epoch)
    writer.add_scalar('LearningRate', scheduler.get_last_lr()[0], epoch)
    
    # Save model checkpoint for last 5 epochs rolling window
    # Save new checkpoint
    torch.save(model.state_dict(), f'/kaggle/working/aneurysm_classifier_epoch_{epoch+1}.pth')
    # Remove old checkpoint
    if epoch >= 5:
        old_epoch = epoch - 5 + 1
        old_checkpoint = f'/kaggle/working/aneurysm_classifier_epoch_{old_epoch}.pth'
        if os.path.exists(old_checkpoint):
            os.remove(old_checkpoint)
    
    # Early stopping if no improvement in metric for 10 epochs
    if epoch == 0:
        best_metric = series_comp_metric
        epochs_no_improve = 0
    else:
        if series_comp_metric > best_metric:
            best_metric = series_comp_metric
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
    if epochs_no_improve >= 10:
        print("Early stopping due to no improvement in metric for 10 epochs.")
        break
    # save current training parameters in case of crash
    with open('training_state.pkl', 'wb') as f:
        pickle.dump({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'best_metric': best_metric,
            'epochs_no_improve': epochs_no_improve
        }, f)
writer.close()

# Average the model checkpoints from the last 5 epochs, compute metric on val set and save the model
for model_epoch in range(max(1, epoch-4), epoch+1):
    checkpoint_path = f'/kaggle/working/aneurysm_classifier_epoch_{model_epoch}.pth'
    if model_epoch == max(1, epoch-4):
        avg_state_dict = torch.load(checkpoint_path)
        for key in avg_state_dict:
            avg_state_dict[key] = avg_state_dict[key].float() / (epoch - max(1, epoch-4) + 1)
    else:
        state_dict = torch.load(checkpoint_path)
        for key in state_dict:
            avg_state_dict[key] += state_dict[key].float() / (epoch - max(1, epoch-4) + 1)
torch.save(avg_state_dict, '/kaggle/working/aneurysm_classifier_averaged.pth')
model.load_state_dict(avg_state_dict)
val_loss, val_metric = test_model(model, val_loader, val_loss_fn)
val_series_metric = test_model_per_series(model, series_path, series_pos_maps, classification_df, batch_size=32)
print(f"Final averaged model - Val batch loss: {val_loss}, Val batch metric: {val_metric}")
print(f"Final averaged model - Val series metric: {val_series_metric}")    

